# Redis Agent Memory data-plane lifecycle

This notebook shows the managed RAM primitives underneath the Agent Framework connector: trusted identities, durable session events, a bounded current window, asynchronous long-term promotion, owner-scoped search, and cleanup.

In [ ]:
#nullable enable
#r "../../src/Redis.Iris.AgentMemory/bin/Release/net8.0/Redis.Iris.AgentMemory.dll"

using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Net;
using System.Net.Http;
using System.Text.Json;
using System.Threading.Tasks;
using Redis.Iris.AgentMemory;

## Load local configuration without displaying secrets

The notebook searches upward for the repository root, reads `.env.local`, and generates an isolated synthetic owner and session.

In [ ]:
DirectoryInfo cursor = new(Environment.CurrentDirectory);
while (cursor is not null && !File.Exists(Path.Combine(cursor.FullName, "ms-agent-framework-redis.slnx"))) cursor = cursor.Parent;
var repoRoot = cursor?.FullName ?? throw new InvalidOperationException("Run this notebook from inside the repository.");
var values = File.ReadLines(Path.Combine(repoRoot, ".env.local"))
    .Where(line => !string.IsNullOrWhiteSpace(line) && !line.TrimStart().StartsWith('#') && line.Contains('='))
    .Select(line => line.Split('=', 2))
    .ToDictionary(parts => parts[0].Trim(), parts => parts[1].Trim().Trim('\"'));
string Required(string name) => values.TryGetValue(name, out var value) && !string.IsNullOrWhiteSpace(value)
    ? value : throw new InvalidOperationException($"Set {name} in .env.local.");

var ownerId = $"csharp-notebook-{Guid.NewGuid():N}";
var sessionId = $"ram-session-{Guid.NewGuid():N}";
var client = new RedisAgentMemoryClient(
    new HttpClient(),
    new() { Endpoint = new Uri(Required("REDIS_IRIS_ENDPOINT")), StoreId = Required("REDIS_IRIS_STORE_ID") },
    new FixedAgentMemoryCredentialProvider(Required("REDIS_IRIS_API_KEY")));

Console.WriteLine($"RAM USER   : {ownerId}");
Console.WriteLine($"RAM SESSION: {sessionId}");

## Write a realistic multi-role session

Each turn is written once. RAM assigns event IDs and server timestamps and retains the ordered session independently of the model context window.

In [ ]:
async Task AddAsync(SessionEventRole role, string actorId, string text)
{
    var item = await client.AddSessionEventAsync(new()
    {
        SessionId = sessionId,
        ActorId = actorId,
        Role = role,
        Content = [new(text)],
        CreatedAt = DateTimeOffset.UtcNow,
        Metadata = JsonSerializer.SerializeToElement(new { sample = "csharp-notebook" }),
    });
    Console.WriteLine($"{item.EventId,-38} {item.Role,-9} {text}");
}

await AddAsync(SessionEventRole.USER, ownerId, "My name is Amelia and I live in Boston.");
await AddAsync(SessionEventRole.ASSISTANT, "travel-agent", "Nice to meet you, Amelia.");
await AddAsync(SessionEventRole.USER, ownerId, "I prefer window seats and morning flights.");
await AddAsync(SessionEventRole.ASSISTANT, "travel-agent", "I will keep those travel preferences in mind.");
await AddAsync(SessionEventRole.USER, ownerId, "I avoid red-eye flights whenever possible.");

## Compare durable session memory with a short-term window

The connector can retain the complete RAM session while providing only the newest events to the model. A managed summary appears separately when the service's summarization threshold is reached.

In [ ]:
var session = await client.GetSessionMemoryAsync(sessionId, includeSummarizedEvents: true);
Console.WriteLine($"SESSION : {session.Events.Count} durable event(s)");
Console.WriteLine(session.Summary is null ? "SUMMARY : not generated yet" : $"SUMMARY : {session.Summary.Text}");
Console.WriteLine("WINDOW  : newest 3 events");
foreach (var item in session.Events.TakeLast(3))
    Console.WriteLine($"  {item.Role,-9} {string.Join(" ", item.Content.Select(part => part.Text))}");

## Search owner-scoped long-term memory

Promotion is asynchronous. Run this cell again after the store's extraction interval if the first result is empty. The equality filter prevents memories belonging to other owners from entering the result set.

In [ ]:
var promoted = await client.SearchLongTermMemoryAsync(new()
{
    Text = "flight seating and schedule preferences",
    Filter = new() { OwnerId = new() { Eq = ownerId } },
    SimilarityThreshold = 0.55,
    Limit = 10,
});
Console.WriteLine($"Found {promoted.Items.Count} promoted memory record(s).");
foreach (var memory in promoted.Items) Console.WriteLine($"- [{memory.Id}] {memory.Text}");

## Owner-scoped cleanup

This removes only the synthetic owner created by this kernel run. Session deletion alone does not remove already promoted long-term memories, so both tiers are handled explicitly.

In [ ]:
var allMemories = await client.SearchLongTermMemoryAsync(new()
{
    Filter = new() { OwnerId = new() { Eq = ownerId } },
    Limit = 100,
});
foreach (var batch in allMemories.Items.Select(memory => memory.Id).Chunk(100))
    await client.BulkDeleteLongTermMemoriesAsync(batch);
await client.DeleteSessionMemoryAsync(sessionId);
Console.WriteLine($"Cleared session {sessionId} and {allMemories.Items.Count} long-term memory record(s).");